In [ ]:
from web3 import Web3
import json
import os

# RPC 및 키 설정
url = f"https://base-mainnet.infura.io/v3/{os.getenv('INFURA_PROJECT_ID')}"

# Attempt to connect to the blockchain
w3 = Web3(Web3.HTTPProvider(url))
PRIVATE_KEY = os.getenv("PRIVATE_KEY")
OWNER = w3.eth.account.from_key(PRIVATE_KEY).address

GAUGE_ADDRESS = '0xF5601F95708256A118EF5971820327F362442D2d'
VOTER_ADDRESS = '0xF5601F95708256A118EF5971820327F362442D2d'
# ABI 파일
gauge_abi = json.load(open("./config/abi/gauge_abi.json"))
voter_abi = json.load(open("./config/abi/voter_abi.json"))

gauge = w3.eth.contract(address=GAUGE_ADDRESS, abi=gauge_abi)
voter = w3.eth.contract(address=VOTER_ADDRESS, abi=voter_abi)

In [2]:
# 1️⃣ 보유 veNFT 총 개수 확인
with open("./config/abi/voting_escrow_abi.json") as f:
    ve_abi = json.load(f)

VE_CONTRACT_ADDRESS = '0xeBf418Fe2512e7E6bd9b87a8F0f294aCDC67e6B4'
ve = w3.eth.contract(address=VE_CONTRACT_ADDRESS, abi=ve_abi)

count = ve.functions.balanceOf(OWNER).call()
count

0

'0x9E70C1eCF5d07f36a6b98664f3eD37c59ccbBEdB'

In [20]:
tokenId = 19705937  # 본인이 보유한 veNFT id
gauges_to_claim = [GAUGE_ADDRESS]  # claim할 gauge 주소 리스트

tx = voter.functions.claimRewards(gauges_to_claim).build_transaction({
    "from": OWNER,
    "nonce": w3.eth.get_transaction_count(OWNER),
    "gasPrice": w3.eth.gas_price,
})
signed = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
tx_hash = w3.eth.send_raw_transaction(signed.rawTransaction)
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Reward claim tx:", tx_hash.hex())


ContractLogicError: ('execution reverted', 'no data')

In [ ]:
gauge.functions.deposit(uint256)

In [12]:
tokenId = '19705937'
tx_hash = gauge.functions.deposit(tokenId).transact({
    "from": OWNER,
    "gas": 200_000,
    "gasPrice": w3.eth.gas_price
})
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Deposited!", receipt.status)


MismatchedABI: 
ABI Not Found!
Found 1 element(s) named `deposit` that accept 1 argument(s).
The provided arguments are not valid.
Provided argument types: (str)
Provided keyword argument types: {}

Tried to find a matching ABI element named `deposit`, but encountered the following problems:
Signature: deposit(uint256), type: function
Argument 1 value `19705937` is not compatible with type `uint256`.


In [ ]:
tx_hash = gauge.functions.deposit(tokenId).transact({
    "from": OWNER,
    "gas": 200_000,
    "gasPrice": w3.eth.gas_price
})
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Deposited!", receipt.status)


AttributeDict({'blockHash': HexBytes('0x4b8713dbf350d70bad962143e844927b6cb91c32a037adecc68abfdbb95c51ea'),
 'blockNumber': 33015960,
 'contractAddress': None,
 'cumulativeGasUsed': 8574909,
 'effectiveGasPrice': 6689872,
 'from': '0x9E70C1eCF5d07f36a6b98664f3eD37c59ccbBEdB',
 'gasUsed': 55437,
 'l1BaseFeeScalar': '0x8dd',
 'l1BlobBaseFee': '0x1',
 'l1BlobBaseFeeScalar': '0x101c12',
 'l1Fee': '0x3fe2407ea',
 'l1GasPrice': '0x1198cea09',
 'l1GasUsed': '0x640',
 'logs': [AttributeDict({'address': '0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913',
   'blockHash': HexBytes('0x4b8713dbf350d70bad962143e844927b6cb91c32a037adecc68abfdbb95c51ea'),
   'blockNumber': 33015960,
   'blockTimestamp': '0x6879ee13',
   'data': HexBytes('0x000000000000000000000000000000000000000000000000000000000052bcfa'),
   'logIndex': 199,
   'removed': False,
   'topics': [HexBytes('0x8c5be1e5ebec7d5bd14f71427d1e84f3dd0314c0f7b2291e5b200ac8c7c3b925'),
    HexBytes('0x0000000000000000000000009e70c1ecf5d07f36a6b98664f3ed3

In [18]:
gauge.functions.deposit(amount).call()

ContractLogicError: ('execution reverted', 'no data')

In [ ]:
# Approve: Gauge 컨트랙트가 LP 토큰을 spend하도록 허용 (ERC20)
erc20_abi = json.load(open("./config/abi/erc20_abi.json"))

lp_token = w3.eth.contract(address='0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913', abi=erc20_abi)
approve_tx = lp_token.functions.approve(
    gauge.address,
    amount
).build_transaction({
    "from": OWNER,
    "nonce": w3.eth.get_transaction_count(OWNER),
    "gasPrice": w3.eth.gas_price,
})

print(approve_tx)

signed_approve = w3.eth.account.sign_transaction(approve_tx, PRIVATE_KEY)
w3.eth.send_raw_transaction(signed_approve.raw_transaction)
w3.eth.wait_for_transaction_receipt(signed_approve.hash)

# Stake (deposit)
    "anonymous": false,
    "inputs": [
      {
        "indexed": true,
        "internalType": "address",
        "name": "user",
        "type": "address"
      },
      {
        "indexed": true,
        "internalType": "uint256",
        "name": "tokenId",
        "type": "uint256"
      },
      {
        "indexed": true,
        "internalType": "uint128",
        "name": "liquidityToStake",
        "type": "uint128"
      }
    ],
    "name": "Deposit",
stake_tx = gauge.functions.deposit(amount).build_transaction({
    "from": OWNER,
    "nonce": w3.eth.get_transaction_count(OWNER),
    "gasPrice": w3.eth.gas_price,
})
signed_stake = w3.eth.account.sign_transaction(stake_tx, PRIVATE_KEY)
tx_hash = w3.eth.send_raw_transaction(signed_stake.raw_transaction)
print("Stake tx hash:", tx_hash.hex())
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Staked!", receipt.status)


ContractLogicError: ('execution reverted', 'no data')